# Exploring the results
Run `memvid analyze -c configs/default.yaml` first. This notebook reads the run outputs; it never changes them.

In [ ]:
import json

import pandas as pd

from memvid.config import load_config

cfg = load_config('../configs/default.yaml')
runs = cfg.video_runs_dir
summary = json.loads((runs / 'analysis' / 'summary.json').read_text())
summary['budgets_tokens']

In [ ]:
rows = []
for p in sorted(runs.glob('*/eval_test.jsonl')):
    rows += [json.loads(x) for x in p.read_text().splitlines()]
df = pd.DataFrame(rows)
df.pivot_table(index='condition', columns='budget', values='correct', aggfunc='mean').round(3)

In [ ]:
df[df.budget.isin(['B2', 'full'])].pivot_table(index='type', columns='condition', values='correct').round(2)

## One question through one run
The operation log shows when the evidence entered the memory and what changed or removed it.

In [ ]:
from memvid.memory.persist import load_memory
from memvid.memory.records import record_overlaps
from memvid.memory.store import parse_record
from memvid.questions.manage import load_questions

q = load_questions(cfg, 'test')[0]
saved = load_memory(runs / 'C3_B2' / 'memory.sqlite')
touched = [o for o in saved.ops
           if o['record'] and record_overlaps(parse_record(o['record']), saved.bounds, q.gt_intervals)]
print(q.question, '->', q.answer)
pd.DataFrame(touched)[['seq', 'chunk_id', 'op', 'label', 'record_id', 'reason']].head(30)